# Extension a cuatro semanas (S1-S4) - ejecucion en Colab

Ventanas de 168 h: **S1** 26 abr-2 may, **S2** 3-9 may, **S3** 10-16 may, **S4** 17-23 may 2026 (semana original). Cada semana entrena solo con datos anteriores a su inicio (`windows.py` recorta los datos al fin de la ventana).
Plan: **54 corridas por semana (216 en total)**, ver `docs/plan_corridas_ventanas.csv`.

## Reparto PC / Colab (no duplicar)
- **PC (script local)**: naive, naive_trend, naive_trend_seasonal, ar, ar_resid_trend_seasonal y **sarimax** (T1 y T3). Columna `ejecuta_en = local` del plan.
- **Colab (este cuaderno)**: xgboost, lightgbm, fcnn, lstm_resid, lstm_direct, ensemble_stl en T1, T2 y T3 (`ejecuta_en = colab`, 168 corridas).
- Los resultados van a `Pipeline_Resultados_Ventanas/<S#>/<RUN_NAME>/` (nunca a `Pipeline_Resultados/`). `run_matrix` **salta** lo ya completo y **reanuda** lo incompleto region por region, asi que relanzar una celda nunca repite trabajo.

## Que ejecutar
1. Entorno de ejecucion: **GPU (T4)**. 2. Celda **1 Setup** (autoriza Drive cuando lo pida). 3. Celda **2 Auditoria** (opcional pero recomendable: 1 minuto). 4. Celda **3 Lanzamiento** y dejarla corriendo. 5. Mas tarde, celdas 4 (comparar S4) y 5 (consolidado).

Dos sesiones en paralelo (dos pestanas/cuentas): en la celda 3 pon `SESION = "A"` en una (S4 y S1) y `SESION = "B"` en la otra (S2 y S3). **Nunca la misma sesion en dos pestanas.**

## 1. Setup

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

!pip install -q optuna lightgbm

REPO_URL = "https://github.com/CarlosT0503/Tesis-forecasting.git"
REPO_DIR = "/content/tesis_repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

DATA_DIR = "/content/drive/MyDrive/Bases de datos Tesis"
ORIG_DIR = "/content/drive/MyDrive/Pipeline_Resultados"
VENT_DIR = "/content/drive/MyDrive/Pipeline_Resultados_Ventanas"
os.makedirs(os.path.join(VENT_DIR, "_auditoria"), exist_ok=True)

from tesis_forecast import windows
from tesis_forecast import ventanas_tools as vt
from tesis_forecast.matrix import run_matrix, resumen_dataframe
print({k: (str(v.test_start), str(v.test_end)) for k, v in windows.VENTANAS.items()})

## 2. Auditoria de datos (solo lectura, ~1 min)

Revisa cobertura por region y semana, GEN/IMP/EXP, y las exogenas globales. Compara las huellas MD5 con `docs/datos_locales_MD5SUMS.txt` del repositorio. Si algo sale raro, **no detiene** la celda 3: solo guarda los CSV en `_auditoria/`.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

dem, ex, gl = vt.auditar_datos_ventanas(DATA_DIR)
for nombre, df in [("auditoria_demanda", dem), ("auditoria_exog_regional", ex), ("auditoria_exog_globales", gl)]:
    df.to_csv(os.path.join(VENT_DIR, "_auditoria", nombre + ".csv"), index=False, encoding="utf-8-sig")

print("Regiones NO aptas para alguna semana:")
print(dem[~dem["apta_para_ventana"].fillna(False)][["ventana", "region", "ultima_fecha", "filas_ventana", "problema"]])
print("
IMP/EXP/GEN - cobertura con valor real (semana S4):")
print(ex[ex["ventana"] == "S4"][["region", "exog", "primera_fecha_con_valor", "ultima_fecha_con_valor", "pct_filas_con_valor"]].to_string(index=False))
print("
Exogenas globales:")
print(gl[gl["ventana"] == "S4"].to_string(index=False))

plan = vt.plan_dataframe()
plan.to_csv(os.path.join(VENT_DIR, "_auditoria", "plan_corridas_ventanas.csv"), index=False, encoding="utf-8-sig")
print("
Plan:", len(plan), "corridas;", plan.groupby("ejecuta_en").size().to_dict())

!cd "{DATA_DIR}" && md5sum *_long.csv *_GEN.csv *_IMP.csv *_EXP.csv IGAE_2.xlsx "Temperaturas promedio.csv"

## 3. LANZAMIENTO (dejar corriendo)

Orden: **T1 -> T2 -> T3**, y dentro de cada nivel una semana tras otra. Asi, si se corta, lo primero que queda completo es lo mas importante (xgboost/lightgbm y las 4 redes con 5 exogenas en todas las semanas) y T3 (variantes de exogenas, 36 corridas por semana) llena el resto.

- Si Colab se desconecta o falla una corrida: **vuelve a correr celda 1 y celda 3**. Lo completo se salta; lo parcial continua desde la ultima region guardada.
- Un fallo en una configuracion no detiene las demas (queda en el resumen y en `tiempos_colab_<SESION>.csv`).
- Tiempos y errores de cada corrida: `Pipeline_Resultados_Ventanas/_auditoria/tiempos_colab_<SESION>.csv`.

In [ ]:
SESION = "A"   # "A": S4 y S1  |  "B": S2 y S3   (una sesion por pestana)
MIS_VENTANAS = {"A": ["S4", "S1"], "B": ["S2", "S3"]}[SESION]
NIVELES = ["T1", "T2", "T3"]
TIEMPOS = os.path.join(VENT_DIR, "_auditoria", f"tiempos_colab_{SESION}.csv")

todos = []
for nivel in NIVELES:
    for v in MIS_VENTANAS:
        configs = vt.filtrar_por_entorno(vt.construir_configs(nivel, v), "colab")
        print(f"
######## {nivel} - {v}: {len(configs)} configuraciones (Colab) ########")
        registros = run_matrix(configs, data_dir=DATA_DIR, mount_drive=False)
        vt.guardar_tiempos(registros, TIEMPOS)
        todos += registros

resumen = resumen_dataframe(todos)
print(resumen["status"].value_counts())
resumen[resumen["status"] == "failed"][["modelo", "ventana", "run_name", "error"]]

## 4. Comparar la reproduccion de S4 con los resultados originales

Correr cuando termine S4 (cualquier momento despues; solo lectura). Las corridas de la PC (naive..., sarimax) aparecen como `FALTA_REPRODUCCION` hasta que copies sus carpetas a Drive (ver `docs/INSTRUCCIONES_EJECUCION.md`). Naive, Naive_Trend y AR salen `NO_COMPARABLE` (su original usaba otra ventana).

Tolerancias propuestas: deterministicos 1e-6, arboles 1e-4, redes +-0.5 pp de MAPE por region.

In [ ]:
cfg_s4 = vt.construir_configs("T1", "S4") + vt.construir_configs("T2", "S4")
detalle, resumen_s4 = vt.comparar_s4_con_original(cfg_s4, ORIG_DIR, VENT_DIR)
resumen_s4

In [ ]:
detalle[~detalle["estado"].isin(["OK"])]

## 5. Consolidado: modelo x region x semana, con medias y desviaciones

In [ ]:
res = vt.consolidar_ventanas(VENT_DIR, out_dir=os.path.join(VENT_DIR, "Consolidado"))
print(res["inventario"].shape)
res["global_"].sort_values("MAPE_media").head(30)